# Biomedical NER All

Model: `d4data/biomedical-ner-all`

This notebook uses a biomedical token-classification model from Hugging Face to detect many clinical and biomedical entity types from input text.

Important note: this model is biomedical NER, but its architecture is `DistilBertForTokenClassification`, not BioBERT. We are using it here because it provides the missing entity categories we wanted to test.

## Supported Entities

This model covers a broad biomedical/clinical label set:

`Activity`, `Administration`, `Age`, `Area`, `Biological_attribute`, `Biological_structure`, `Clinical_event`, `Color`, `Coreference`, `Date`, `Detailed_description`, `Diagnostic_procedure`, `Disease_disorder`, `Distance`, `Dosage`, `Duration`, `Family_history`, `Frequency`, `Height`, `History`, `Lab_value`, `Mass`, `Medication`, `Nonbiological_location`, `Occupation`, `Other_entity`, `Other_event`, `Outcome`, `Personal_background`, `Qualitative_concept`, `Quantitative_concept`, `Severity`, `Sex`, `Shape`, `Sign_symptom`, `Subject`, `Texture`, `Therapeutic_procedure`, `Time`, `Volume`, `Weight`.

The model uses BIO labels internally, such as `B-Disease_disorder`, `I-Disease_disorder`, and `O`. The notebook aggregates those token labels into readable spans.

In [7]:
from __future__ import annotations

import pandas as pd
import torch
from transformers import AutoModelForTokenClassification, AutoTokenizer, pipeline

MODEL_NAME = "d4data/biomedical-ner-all"
device = 0 if torch.cuda.is_available() else -1

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForTokenClassification.from_pretrained(MODEL_NAME)

ner_pipeline = pipeline(
    task="token-classification",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="simple",
    device=device,
)

print(f"Loaded {MODEL_NAME} on {'cuda' if device == 0 else 'cpu'}.")
print("Raw BIO labels:")
print(model.config.id2label)

Device set to use cpu


Loaded d4data/biomedical-ner-all on cpu.
Raw BIO labels:
{0: 'O', 1: 'B-Activity', 2: 'B-Administration', 3: 'B-Age', 4: 'B-Area', 5: 'B-Biological_attribute', 6: 'B-Biological_structure', 7: 'B-Clinical_event', 8: 'B-Color', 9: 'B-Coreference', 10: 'B-Date', 11: 'B-Detailed_description', 12: 'B-Diagnostic_procedure', 13: 'B-Disease_disorder', 14: 'B-Distance', 15: 'B-Dosage', 16: 'B-Duration', 17: 'B-Family_history', 18: 'B-Frequency', 19: 'B-Height', 20: 'B-History', 21: 'B-Lab_value', 22: 'B-Mass', 23: 'B-Medication', 24: 'B-Non[biological](Detailed_description', 25: 'B-Nonbiological_location', 26: 'B-Occupation', 27: 'B-Other_entity', 28: 'B-Other_event', 29: 'B-Outcome', 30: 'B-Personal_[back](Biological_structure', 31: 'B-Personal_background', 32: 'B-Qualitative_concept', 33: 'B-Quantitative_concept', 34: 'B-Severity', 35: 'B-Sex', 36: 'B-Shape', 37: 'B-Sign_symptom', 38: 'B-Subject', 39: 'B-Texture', 40: 'B-Therapeutic_procedure', 41: 'B-Time', 42: 'B-Volume', 43: 'B-Weight', 44

## Input Text

Change `input_text` and rerun the cells to test another sentence.

In [13]:
input_text = "On 03/12/2019, Dr. Emily Smith at St. Mary's Hospital in Stuttgart prescribed ibuprofen 400 mg to 67-year-old Mr. David Jones, patient ID MRN-45892, for severe left knee pain caused by osteoarthritis after an X-ray showed joint-space narrowing."

input_text

"On 03/12/2019, Dr. Emily Smith at St. Mary's Hospital in Stuttgart prescribed ibuprofen 400 mg to 67-year-old Mr. David Jones, patient ID MRN-45892, for severe left knee pain caused by osteoarthritis after an X-ray showed joint-space narrowing."

## Run NER

In [14]:
raw_entities = ner_pipeline(input_text)

pd.DataFrame(raw_entities)

,entity_group,score,word,start,end
0,Date,0.995903,03 / 12 / 2019,3,13
1,Nonbiological_location,0.973636,st. mary ' s hospital in,34,56
2,Nonbiological_location,0.982329,stuttgart,57,66
3,Medication,0.954525,ibuprofen,78,87
4,Dosage,0.999715,400 mg,88,94
5,Age,0.994509,67 - year - old,98,109
6,Sex,0.577200,mr,110,112
7,Severity,0.999819,severe,153,159
8,Biological_structure,0.999934,left knee,160,169
9,Sign_symptom,0.999962,pain,170,174


## Cleaned Entity Spans

This table keeps the entity type, detected text, character offsets, and confidence score.

In [15]:
def clean_entity_spans(entities: list[dict]) -> pd.DataFrame:
    rows = []
    for entity in entities:
        label = entity.get("entity_group") or entity.get("entity", "")
        label = label.removeprefix("B-").removeprefix("I-")
        rows.append(
            {
                "entity": label,
                "text": entity["word"].replace(" ##", ""),
                "start": entity["start"],
                "end": entity["end"],
                "score": round(float(entity["score"]), 4),
            }
        )

    return pd.DataFrame(rows, columns=["entity", "text", "start", "end", "score"])


cleaned_entities = clean_entity_spans(raw_entities)
cleaned_entities

,entity,text,start,end,score
0,Date,03 / 12 / 2019,3,13,0.9959
1,Nonbiological_location,st. mary ' s hospital in,34,56,0.9736
2,Nonbiological_location,stuttgart,57,66,0.9823
3,Medication,ibuprofen,78,87,0.9545
4,Dosage,400 mg,88,94,0.9997
5,Age,67 - year - old,98,109,0.9945
6,Sex,mr,110,112,0.5772
7,Severity,severe,153,159,0.9998
8,Biological_structure,left knee,160,169,0.9999
9,Sign_symptom,pain,170,174,1.0000


## All Detected Entities

This view uses every entity label predicted by the model. No entity categories are filtered out.

In [16]:
all_detected_entities = cleaned_entities.reset_index(drop=True)
all_detected_entities

,entity,text,start,end,score
0,Date,03 / 12 / 2019,3,13,0.9959
1,Nonbiological_location,st. mary ' s hospital in,34,56,0.9736
2,Nonbiological_location,stuttgart,57,66,0.9823
3,Medication,ibuprofen,78,87,0.9545
4,Dosage,400 mg,88,94,0.9997
5,Age,67 - year - old,98,109,0.9945
6,Sex,mr,110,112,0.5772
7,Severity,severe,153,159,0.9998
8,Biological_structure,left knee,160,169,0.9999
9,Sign_symptom,pain,170,174,1.0000


## All Unique Entity Types Predicted

Use this to quickly see which entity categories appeared in the current sentence.

In [12]:
sorted(cleaned_entities["entity"].unique()) if not cleaned_entities.empty else []

['Biological_structure',
 'Date',
 'Medication',
 'Nonbiological_location',
 'Severity']